# Libraries

In [1]:
#!pip install -q torch==2.9.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126
!pip -q install "lm_eval[hf]"
!pip -q install "lm_eval[math]" #for MATH-500
!pip -q install --upgrade transformers
!pip -q install langdetect --break-system-packages #for If-eval Task

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.4/56.4 kB 3.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 103.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.1/91.1 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.2/144.2 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.1/209.1 kB 13.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
omegaconf 2.3.0 requires antlr4-python3-runtime==4.9.*, but you have antlr4-python3-runtime 4.11.0 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB

In [2]:
# Libraries
import gc
import glob
import shutil
import os
import random
import subprocess

import numpy as np
import torch
import lm_eval

from dataclasses import dataclass
from typing import Optional
from huggingface_hub import login
from IPython.display import FileLink
from kaggle_secrets import UserSecretsClient

# Global Variables

In [3]:
MODEL_NAME       = "Llama-3.2-3B-Instruct-SparseGPT-70"
MODEL_PRETRAINED = "EdgeCompress01/Llama-3.2-3B-Instruct-SparseGPT"
SUB_FOLDER = "Models/70"
SEED             = 42

os.environ["HF_ALLOW_CODE_EVAL"] = "1"

# Classes

In [4]:
@dataclass
class Task:
    key:               str
    name:              str
    category:          str
    num_fewshot:       int  = 0
    batch_size:        int  = 2
    limit:             Optional[int] = None
    apply_chat_template: bool = True
    max_gen_toks:      Optional[int] = None    

# Functions

In [5]:
# REPRODUCIBILITY
def set_reproducibility(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # Ensure deterministic behavior in CuDNN
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"Global seed set to {seed}")

# Seeding

In [6]:
set_reproducibility(SEED)

Global seed set to 42


# Huggingface login

In [7]:
# HUGGING FACE AUTHENTICATION
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)
print("Logging into Hugging Face...")

Logging into Hugging Face...


# Evaluations

**Configurations**

In [8]:
# Task definition
TASKS = [
    # Math
     ##Task("math-500",     "minerva_math500",           "math",                num_fewshot=0, batch_size=8, max_gen_toks=2048),
     ##Task("gsm8k",        "gsm8k_cot",                 "math",                num_fewshot=8, batch_size=6,  max_gen_toks=1024),

    # Reasoning
     ##Task("gpqa_diamond", "gpqa_diamond_cot_zeroshot", "reasoning",           batch_size=8,  max_gen_toks=2048),
     ##Task("arc_challenge","arc_challenge",              "reasoning",           batch_size=8),

    # Instruction Following
     Task("ifeval",       "ifeval",                    "instruction_following",batch_size=6,  max_gen_toks=1024),

    # Perplexity
     Task("wikitext",     "wikitext",                  "perplexity",          batch_size=1,  apply_chat_template=False),

    # Knowledge
     Task("mmlu",         "mmlu",                      "knowledge",           num_fewshot=5, batch_size=2,  limit=1400),
]


# Model args
MODEL_ARGS = ",".join([
    f"pretrained={MODEL_PRETRAINED}",
    f"subfolder={SUB_FOLDER}",
    "dtype=float16",
    "parallelize=True",
    "max_length=4096",
    "trust_remote_code=True",
    #"enable_thinking=False",
])

results_dir = f"evaluation_results/{MODEL_NAME}"

**Evaluation loop**

In [9]:
for t in TASKS:
    print(f"\n[{t.category.upper()}] {t.key}")

    cmd = [
        "lm_eval",
        "--model",       "hf",
        "--model_args",  MODEL_ARGS,
        "--tasks",       t.name,
        "--num_fewshot", str(t.num_fewshot),
        "--batch_size",  str(t.batch_size),
        "--seed",        str(SEED),
        "--verbosity", "WARNING",
        "--output_path", f"{results_dir}/{t.key}",
    ]

    if t.limit is not None:
        cmd += ["--limit", str(t.limit)]
        
    if t.apply_chat_template:
            cmd.append("--apply_chat_template")
        
    if t.max_gen_toks is not None:                             
        cmd += ["--gen_kwargs", f"max_gen_toks={t.max_gen_toks}"]

    subprocess.run(cmd, check=True)

    torch.cuda.empty_cache()
    gc.collect()
    print("Done")


[INSTRUCTION_FOLLOWING] ifeval


2026-04-08:16:11:46 INFO     [config.evaluate_config:301] Using default fewshot_as_multiturn=True.
2026-04-08:16:11:52 INFO     [_cli.run:376] Selected Tasks: ['ifeval']
2026-04-08:16:11:54 WARNING  [evaluator:223] generation_kwargs: {'max_gen_toks': 1024} specified through cli, these settings will update set parameters in yaml tasks. Ensure 'do_sample=True' for non-greedy decoding!
Generating train split: 100%|██████████| 541/541 [00:00<00:00, 9932.32 examples/s]
2026-04-08:16:12:56 WARNING  [evaluator:490] Chat template formatting change affects loglikelihood and multiple-choice tasks. See docs/chat-template-readme.md for details.
Running generate_until requests: 100%|██████████| 541/541 [1:38:11<00:00, 10.89s/it]
ERROR:root:Unable to detect language for text ** ** ** ** ** ** 

 ** ** ** ** ** 

 ** ** ** ** ** ** 

 ** ** ** 

 ** ** ** 

 ** ** ** 

 ** ** ** ** 

 ** ** ** 

 ** ** ** 

 ** ** ** ** 

 ** ** ** ** 

 ** ** ** 

 ** ** ** ** 

 ** ** ** ** 

 ** ** ** ** 

 ** ** 

hf ({'pretrained': 'EdgeCompress01/Llama-3.2-3B-Instruct-SparseGPT', 'subfolder': 'Models/70', 'dtype': 'float16', 'parallelize': True, 'max_length': 4096}), gen_kwargs: ({'max_gen_toks': 1024}), limit: None, num_fewshot: 0, batch_size: 6
|Tasks |Version|Filter|n-shot|        Metric         |   |Value |   |Stderr|
|------|------:|------|-----:|-----------------------|---|-----:|---|------|
|ifeval|      4|none  |     0|inst_level_loose_acc   |↑  |0.2398|±  |   N/A|
|      |       |none  |     0|inst_level_strict_acc  |↑  |0.2314|±  |   N/A|
|      |       |none  |     0|prompt_level_loose_acc |↑  |0.1275|±  |0.0144|
|      |       |none  |     0|prompt_level_strict_acc|↑  |0.1183|±  |0.0139|

Done

[PERPLEXITY] wikitext


2026-04-08:17:51:23 INFO     [_cli.run:376] Selected Tasks: ['wikitext']
2026-04-08:17:51:23 WARNING  [evaluator:181] pretrained=EdgeCompress01/Llama-3.2-3B-Instruct-SparseGPT appears to be an instruct or chat variant but chat template is not applied.
        Recommend setting `apply_chat_template` (optionally `fewshot_as_multiturn`).
Loading weights: 100%|██████████| 254/254 [00:01<00:00, 151.77it/s]
2026-04-08:17:51:34 WARNING  [api.task:729] [Task: wikitext] metric word_perplexity is defined, but aggregation is not. using default aggregation=weighted_perplexity
2026-04-08:17:51:34 WARNING  [api.task:741] [Task: wikitext] metric word_perplexity is defined, but higher_is_better is not. using default higher_is_better=False
2026-04-08:17:51:34 WARNING  [api.task:729] [Task: wikitext] metric byte_perplexity is defined, but aggregation is not. using default aggregation=weighted_perplexity
2026-04-08:17:51:34 WARNING  [api.task:741] [Task: wikitext] metric byte_perplexity is defined, but h

hf ({'pretrained': 'EdgeCompress01/Llama-3.2-3B-Instruct-SparseGPT', 'subfolder': 'Models/70', 'dtype': 'float16', 'parallelize': True, 'max_length': 4096}), gen_kwargs: ({}), limit: None, num_fewshot: 0, batch_size: 1
| Tasks  |Version|Filter|n-shot|    Metric     |   |  Value  |   |Stderr|
|--------|------:|------|-----:|---------------|---|--------:|---|------|
|wikitext|      2|none  |     0|bits_per_byte  |↓  |   2.2126|±  |   N/A|
|        |       |none  |     0|byte_perplexity|↓  |   4.6350|±  |   N/A|
|        |       |none  |     0|word_perplexity|↓  |3644.8032|±  |   N/A|

Done

[KNOWLEDGE] mmlu


2026-04-08:17:57:51 WARNING  [config.evaluate_config:281] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-04-08:17:57:51 INFO     [config.evaluate_config:301] Using default fewshot_as_multiturn=True.
2026-04-08:17:57:56 INFO     [_cli.run:376] Selected Tasks: ['mmlu']
Generating dev split: 100%|██████████| 5/5 [00:00<00:00, 2646.58 examples/s]
2026-04-08:17:59:09 WARNING  [evaluator:333] Overwriting default num_fewshot of mmlu_abstract_algebra from None to 5
2026-04-08:17:59:09 WARNING  [evaluator:333] Overwriting default num_fewshot of mmlu_anatomy from None to 5
2026-04-08:17:59:09 WARNING  [evaluator:333] Overwriting default num_fewshot of mmlu_astronomy from None to 5
2026-04-08:17:59:09 WARNING  [evaluator:333] Overwriting default num_fewshot of mmlu_college_biology from None to 5
2026-04-08:17:59:09 WARNING  [evaluator:333] Overwriting default num_fewshot of mmlu_college_chemistry from None to 5
2026-04-08:17:59:09 WARNING  [evaluato

hf ({'pretrained': 'EdgeCompress01/Llama-3.2-3B-Instruct-SparseGPT', 'subfolder': 'Models/70', 'dtype': 'float16', 'parallelize': True, 'max_length': 4096}), gen_kwargs: ({}), limit: 1400.0, num_fewshot: 5, batch_size: 2
|                 Tasks                 |Version|Filter|n-shot|Metric|   |Value |   |Stderr|
|---------------------------------------|------:|------|-----:|------|---|-----:|---|-----:|
|mmlu                                   |      2|none  |      |acc   |↑  |0.2281|±  |0.0036|
| - humanities                          |      2|none  |      |acc   |↑  |0.2398|±  |0.0063|
|  - formal_logic                       |      1|none  |     5|acc   |↑  |0.2619|±  |0.0393|
|  - high_school_european_history       |      1|none  |     5|acc   |↑  |0.2182|±  |0.0323|
|  - high_school_us_history             |      1|none  |     5|acc   |↑  |0.2500|±  |0.0304|
|  - high_school_world_history          |      1|none  |     5|acc   |↑  |0.2447|±  |0.0280|
|  - international_law             

# Save reports

In [10]:
zip_path = shutil.make_archive(
    "evaluation_results",   # zip file name
    "zip",                  # format
    "evaluation_results"    # folder to compress
)